**1.普通对话**

调用本地 Ollama 模型

In [ ]:
from ollama import chat

#最基本最简单的代码
response = chat(
    model = "qwen2.5:7b",
    messages = [
        {
            "role": "user",
            "content": "你好，请介绍一下你自己。"
        }
    ]
)

print(response["message"]["content"])

你好！我是Qwen，由阿里云开发的人工智能模型。我能够帮助用户生成各种类型的文本内容，比如文章、故事、诗歌、故事等，并能根据不同的场景和需求进行变换和扩展。同时，我也支持多语言交流，致力于为全球用户提供便捷的服务。如果你有任何问题或需要帮助，都可以随时告诉我哦！


In [ ]:
#能连续聊天
from ollama import chat

messages = [] #chat history

while True:
    user_input = input("You: ")

    if user_input == "exit":
        break

    messages.append({
        "role": "user",
        "content": user_input
    })

    response = chat(
        model="qwen2.5:7b",
        messages=messages
    )

    assistant_reply = response["message"]["content"]

    print("AI:", assistant_reply)

    messages.append({
        "role": "assistant",
        "content": assistant_reply
    })


调用云端 DeepSeek API

In [ ]:
import os
from openai import OpenAI

api_key = os.getenv("DEEPSEEK_API_KEY")

client = OpenAI(
    api_key=api_key,
    base_url="https://api.deepseek.com"
)

response = client.chat.completions.create(
    model="deepseek-chat",
    messages=[
        {
            "role": "user",
            "content": "你好"
        }
    ]
)

print(response.choices[0].message.content)

Hello! How can I assist you today?


In [ ]:
#连续聊天
from openai import OpenAI

client = OpenAI(
    api_key= api_key,
    base_url="https://api.deepseek.com"
)

messages = []

while True:
    question = input("You: ")

    if question == "exit":
        break

    messages.append({
        "role": "user",
        "content": question
    })

    response = client.chat.completions.create(
        model="deepseek-chat",
        messages=messages
    )

    answer = response.choices[0].message.content

    print("AI:", answer)

    messages.append({
        "role": "assistant",
        "content": answer
    })

**2. System Prompt**
    - 学会通过 `role="system"` 控制模型行为（例如让模型扮演翻译、代码助手等）

In [8]:
import os
from openai import OpenAI

api_key = os.getenv("DEEPSEEK_API_KEY")

client = OpenAI(
    api_key=api_key,
    base_url="https://api.deepseek.com"
)

response = client.chat.completions.create(
    model="deepseek-chat",
    messages=[
        {    
            "role": "system",
            "content": "以后所有回答都使用英文。"
        },
        {
            "role": "user",
            "content": "你好"
        }
    ]
)

print(response.choices[0].message.content)

Hello! How can I assist you today?


**以上例子系统输出就会变成英文**
*system prompt用途：1.指定身份（模型会尽量按照这个身份去回答） 2. 指定输出风格 3.指定语言 4.指定规则*
不是用户说的话，用户不能直接看到它

System放：

身份（Role）
行为规则（Rules）
长期要求（Persistent Instructions）
输出风格（Style）
输出语言（Language）
安全限制（Constraints）

User放：

当前任务（Current Task）
当前问题（Question）
当前需求（Request）

**3.会让模型输出结构化json**

第一种：system prompt

In [ ]:
from openai import OpenAI
import os
import json

api_key = os.getenv("DEEPSEEK_API_KEY")

client = OpenAI(
    api_key=api_key,
    base_url="https://api.deepseek.com"
)

response = client.chat.completions.create(
    model="deepseek-chat",
    messages=[
        {
            "role":"system",
            "content":"""
请只返回 JSON，不要输出任何解释。

格式：

{
    "language":"",
    "description":""
}
"""
        },
        {
            "role":"user",
            "content":"介绍一下Python。"
        }
    ]
)

content = response.choices[0].message.content

print(content)
#把字符串转换为JSON对象，data就变成了一个字典，可以通过键来访问对应的值
data = json.loads(content)

print(data["language"])
print(data["description"])

{
    "language":"Python",
    "description":"Python是一种高级、通用、解释型的编程语言，以其简洁易读的语法和强大的标准库而闻名，广泛应用于Web开发、数据分析、人工智能和自动化等领域。"
}
Python
Python是一种高级、通用、解释型的编程语言，以其简洁易读的语法和强大的标准库而闻名，广泛应用于Web开发、数据分析、人工智能和自动化等领域。


JSON（JavaScript Object Notation）就是一种数据格式。

你可以把它理解成：

Python 字典的字符串版本

注意：模型输出的是字符串

json.loads() 的作用是把JSON 字符串转换成 Python 字典。

第二种：JSON Schema

In [11]:
from ollama import chat

schema = {
    "type": "object",
    "properties": {
        "name": {
            "type": "string"
        },
        "age": {
            "type": "integer"
        },
        "job": {
            "type": "string"
        }
    },
    "required": [
        "name",
        "age",
        "job"
    ]
}

response = chat(
    model="qwen2.5:7b",
    messages=[
        {
            "role": "user",
            "content": "介绍一下爱因斯坦"
        }
    ],
    format=schema
)

print(response["message"]["content"])

{
    "name": "阿尔伯特·爱因斯坦",
    "age": 0,
    "job": "理论物理学家、哲学家和和平主义者，20世纪最伟大的科学家之一。被广泛认为是相对论的创立者，并因此而闻名于世。他也是原子能研究的先驱者之一，其贡献推动了核物理学的发展。此外，他还因其在科学界以外的影响力而广受尊敬，在公共场合积极支持反战、和平及社会正义等事业，是一位有着深厚人文关怀的科学家。他的一生不仅致力于科学研究，也积极参与各种社会活动与公益活动。他的理论改变了我们对宇宙的理解，并启发了无数科学家继续探索未知领域。对于爱因斯坦而言，科学不仅仅是研究客观世界的现象和规律，更是一种追求真理、热爱自由的生活态度。1921年，爱因斯坦因其在光电效应方面的贡献而获得了诺贝尔物理学奖；1955年4月18日，在瑞士的家中去世，享年76岁。"
}


第三种：Pydantic (最推荐)

In [ ]:
import ollama
from pydantic import BaseModel


# 让模型输出结构化JSON —— 用format参数传入schema，模型被强制按这个结构回答
class TaskInfo(BaseModel):
    task: str
    priority: str

r2 = ollama.chat(
    model="qwen2.5:7b",
    messages=[{"role": "user", "content": "结构化这句话：明天必须交完Agent作业，很急"}],
    format=TaskInfo.model_json_schema(),
    options={"temperature": 0},
)
print(TaskInfo.model_validate_json(r2.message.content))

我是Qwen，来自阿里云，能与你进行自然流畅的对话。
task='结构化句子' priority='紧急'


**3.会定义一个工具函数**

In [ ]:
from ollama import chat
import json

response = chat(
    model="qwen2.5:7b",
    messages=[
        {
            "role": "system",
            "content": """
判断用户是否需要计算器。

如果需要返回：
tool名称和表达式。
"""
        },
        {
            "role": "user",
            "content": "123乘456是多少"
        }
    ],
    format=schema
)

print(response["message"]["content"])


能力：模型自主选择工具 -> 执行工具 -> 结果喂回模型 -> 给出最终回答

In [5]:

import time
import ollama
import threading

MODEL = "qwen2.5:7b"


# ---------- 工具定义 ----------

def calculator(expression: str) -> str:
    """计算一个只包含数字和 + - * / ( ) 的数学表达式"""
    allowed = set("0123456789+-*/(). ")
    if not all(c in allowed for c in expression):
        return "错误：表达式包含不允许的字符"
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"计算出错: {e}"


def read_file(path: str) -> str:
    """读取本地文本文件内容"""
    try:
        with open(path, "r", encoding="utf-8") as f:
            return f.read()[:2000]
    except Exception as e:
        return f"读取文件出错: {e}"


def search(query: str) -> str:
    """在本地小知识库中按关键词搜索（占位数据，以后可换成真正的向量检索）"""
    fake_kb = {
        "python": "Python是一种解释型、动态类型的编程语言。",
        "agent": "Agent是能感知环境、自主决策并执行动作的程序。",
        "ollama": "Ollama是一个可以在本地运行大语言模型的工具。",
    }
    for k, v in fake_kb.items():
        if k in query.lower():
            return v
    return "知识库中没有找到相关内容"


# 把工具描述成 JSON Schema 告诉模型
TOOL_FUNCTIONS = {"calculator": calculator, "read_file": read_file, "search": search}

TOOLS_SCHEMA = [
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "计算一个数学表达式并返回结果",
            "parameters": {
                "type": "object",
                "properties": {"expression": {"type": "string", "description": "要计算的表达式"}},
                "required": ["expression"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "read_file",
            "description": "读取本地文件的文本内容",
            "parameters": {
                "type": "object",
                "properties": {"path": {"type": "string", "description": "文件路径"}},
                "required": ["path"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "search",
            "description": "在知识库中搜索相关信息",
            "parameters": {
                "type": "object",
                "properties": {"query": {"type": "string", "description": "搜索关键词"}},
                "required": ["query"],
            },
        },
    },
]

def call_model_with_timeout(messages: list, timeout: int):
    result = [None]
    error = [None]

    def target():
        try:
            result[0] = ollama.chat(model=MODEL, messages=messages, tools=TOOLS_SCHEMA)
        except Exception as e:
            error[0] = e

    thread = threading.Thread(target=target)
    thread.start()
    thread.join(timeout=timeout)

    if thread.is_alive():
        return None, "模型响应超时，已终止"
    if error[0]:
        return None, f"调用模型出错: {error[0]}"
    return result[0], None

# ---------- Agent Loop：选工具 -> 执行工具 -> 喂回结果 -> 返回最终答案 ----------

def run_agent(user_input: str, max_steps: int = 5, timeout_per_call: int = 30) -> str:
    messages = [{"role": "user", "content": user_input}]
    print(f"\n🧑 用户输入: {user_input}\n{'='*50}")

    for step in range(1, max_steps + 1):
        print(f"\n--- Step {step} ---")
        response, err = call_model_with_timeout(messages, timeout=timeout_per_call)
        if err:
            return err

        msg = response.message
        messages.append(msg)

        if not msg.tool_calls:
            print(f"✅ 模型直接回答（无工具调用）")
            return msg.content

        # ↓ 这里就是你缺的部分
        print(f"🔧 模型决定调用 {len(msg.tool_calls)} 个工具:")
        for tool_call in msg.tool_calls:
            name = tool_call.function.name
            args = tool_call.function.arguments
            print(f"   → {name}({args})")

            func = TOOL_FUNCTIONS.get(name)
            try:
                result = func(**args) if func else f"未知工具: {name}"
            except Exception as e:
                result = f"工具执行出错: {e}"

            print(f"   ← 结果: {result}")
            messages.append({"role": "tool", "name": name, "content": str(result)})

    return "已达到最大步数，仍未得到最终答案"

if __name__ == "__main__":
    answer = run_agent("帮我算一下 (23 + 7) / 0，然后告诉我什么是Ollama")
    print("最终答案:", answer)


🧑 用户输入: 帮我算一下 (23 + 7) / 0，然后告诉我什么是Ollama

--- Step 1 ---
🔧 模型决定调用 1 个工具:
   → search({'query': '什么是Ollama'})
   ← 结果: Ollama是一个可以在本地运行大语言模型的工具。

--- Step 2 ---
✅ 模型直接回答（无工具调用）
最终答案: 表达式 \((23 + 7) / 0\) 在数学中是未定义的，因为除以零没有意义。

关于 Ollama，它是一个可以在本地运行的大语言模型的工具。如果您有关于这些内容的进一步问题，请随时告诉我！


模型行为不稳定，
同一个输入，多跑几次你会得到不同的 tool call 顺序和组合。这在没有 system prompt 约束的情况下是正常的。
可以加上 system prompt 来稳定行为